# [LAB] LangGraph 심화 실습

그래프를 나누고(Subgraph), 상태를 저장하고(Checkpointer, Memory Store), 여러 작업을 동시에 실행하는(Fan-out/Fan-in, Send) 완성된 코드를 실행해 봅니다. `# ✍️ CHANGE_HERE:` 자리에서는 값을 바꿔 다시 실행하고, 그래프의 동작이 어떻게 달라지는지 봅니다.

- **2장 Subgraph로 요약·번역 나누기**: 요약·번역 그래프를 따로 만들어 부모 그래프의 노드 안에서 호출합니다.
- **3장 MemorySaver로 대화 이어 가기**: 체크포인터를 붙여 앞에서 한 말을 기억하게 합니다.
- **4장 SqliteSaver로 저장하고 되돌아가기**: 대화 기록을 파일에 남기고, 지난 시점으로 되돌아가 다시 실행합니다.
- **5장 Memory Store로 대화를 넘어 기억하기**: 새 대화에서도 사용자의 이름과 취미를 기억하게 합니다.
- **6장 장점과 단점을 동시에 정리하기**: 두 노드를 동시에 실행하고 한 노드에서 합칩니다.
- **7장 질문 수만큼 동시에 답하기**: 질문 목록 길이만큼 작업을 만들어 동시에 실행합니다.

셀은 위에서부터 차례로 실행합니다. 각 장 첫머리의 접힌 준비 코드도 실행해야 그 아래 셀이 동작합니다. 모든 장이 LLM을 호출하므로 결과가 나오기까지 수십 초가 걸릴 수 있고, 답은 실행할 때마다 조금씩 다릅니다.

**목차**

- 1. 실습 방법
- 2. Subgraph로 요약·번역 나누기
  - 2.1 요약·번역 Subgraph
  - 2.2 부모 그래프의 State와 노드
  - 2.3 부모 그래프 연결
  - 2.4 실행하기
- 3. MemorySaver로 대화 이어 가기
  - 3.1 State와 chatbot 노드
  - 3.2 그래프 연결
  - 3.3 MemorySaver로 컴파일하고 실행하기
- 4. SqliteSaver로 저장하고 되돌아가기
  - 4.1 SqliteSaver로 저장하기
  - 4.2 체크포인트 목록 보기
  - 4.3 과거 체크포인트로 되돌아가기
- 5. Memory Store로 대화를 넘어 기억하기
  - 5.1 장기 기억 Store 만들기
  - 5.2 기억을 읽고 쓰는 chatbot 노드
  - 5.3 그래프 연결과 컴파일
  - 5.4 대화 함수
  - 5.5 두 세션으로 대화하기
- 6. 장점과 단점을 동시에 정리하기
  - 6.1 장점·단점·결론 노드
  - 6.2 병렬로 연결하기
  - 6.3 실행하기
- 7. 질문 수만큼 동시에 답하기
  - 7.1 State와 worker 노드
  - 7.2 fan_out 함수
  - 7.3 동적으로 연결하기
  - 7.4 실행하기
- 8. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env()

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 ⛔ 줄의 → 안내대로 `.env`를 고칩니다. `.env`는 노트북을 시작할 때 한 번만 읽으므로, 고친 뒤에는 커널을 다시 시작하고 맨 위 셀부터 다시 실행합니다. 무엇을 고칠지 모르겠으면 그 셀의 출력을 AI에게 붙여 넣고 물어봅니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 셀 출력을 지우고, 그 내용을 다른 곳에 붙여 넣지 않습니다.

LangChain 디버그 모드를 켭니다. 체인과 그래프가 실행될 때 단계마다 입력과 출력이 셀 아래에 출력됩니다.

In [ ]:
from langchain_core.globals import set_debug

# 출력이 너무 길면 True를 False로 바꾸고 이 셀을 다시 실행합니다
set_debug(True)

## 1. 실습 방법

코드는 모두 완성되어 있습니다. 위에서부터 차례로 실행합니다.

- `# ✍️ CHANGE_HERE:` 주석이 있는 셀은 먼저 그대로 실행해 결과를 봅니다. 그다음 셀 위의 **바꿔 볼 값**을 복사해, 주석 바로 아래 줄의 표시된 자리(지금 값)를 지우고 붙여 넣습니다. 어떻게 달라질지 생각해 본 뒤 다시 실행합니다.
- 달라진 결과를 본 뒤 셀 아래 접힌 **무슨 일이 생기나**를 펼쳐 비교합니다. 확인했으면 값을 원래대로 되돌립니다.

**막혔을 때 AI에게 도움받는 프롬프트 힌트** 오류가 난 셀의 코드와 오류 메시지 전체를 붙여 넣고, 원인과 고칠 곳을 쉬운 말로 알려 달라고 합니다.

## 2. Subgraph로 요약·번역 나누기

문서를 받아 요약하고 영어로 옮긴 뒤, 그 요약을 바탕으로 질문에 답하는 그래프를 만듭니다. 요약·번역은 따로 만든 작은 그래프(Subgraph)가 맡고, 부모 그래프는 노드 하나에서 그 Subgraph를 호출합니다.

두 그래프의 State는 다릅니다. Subgraph는 `text`(원문)와 `summary`(요약)만 알고, 부모는 `messages`(대화), `document`(문서), `summary`(영문 요약)를 가집니다. 부모 노드가 값을 옮겨 넣고 받아 와야 합니다.

그래프는 맨 아래 "실행하기" 셀에서 처음 실행됩니다. 위의 셀을 차례로 실행해 두고, 마지막 셀에서 결과를 확인합니다.

준비 코드입니다. 필요한 도구를 불러오고 `llm = get_llm()`으로 모델을 준비합니다. 접혀 있으니 펼치지 않아도 되고, 실행만 합니다.

In [ ]:
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

### 2.1 요약·번역 Subgraph

Subgraph에 노드 두 개를 등록하고 차례로 잇습니다. `summarize`가 원문을 한 문장으로 요약하고, `translate`가 그 요약을 영어로 옮깁니다. 바로 아래 줄이 이 그래프를 따로 컴파일해 `summary_subgraph`로 만듭니다.

In [ ]:
# ── 서브그래프 ────────────────────────────────────────────────────────────────
# 서브그래프는 독립적으로 컴파일된 그래프로, 부모 그래프의 노드로 사용된다.
# 복잡한 워크플로우를 재사용 가능한 단위로 분리할 때 활용한다.


class SummaryState(TypedDict):
    """서브그래프 전용 State. 부모 State와 독립적이다."""

    text: str
    summary: str


def summarize_node(state: SummaryState) -> dict:
    result = llm.invoke(f"다음 텍스트를 한 문장으로 요약해줘:\n\n{state['text']}")
    return {"summary": result.content}


def translate_node(state: SummaryState) -> dict:
    result = llm.invoke(f"다음 문장을 영어로 번역해줘:\n\n{state['summary']}")
    return {"summary": result.content}


summary_builder = StateGraph(SummaryState)
summary_builder.add_node("summarize", summarize_node)
summary_builder.add_node("translate", translate_node)
summary_builder.add_edge(START, "summarize")
summary_builder.add_edge("summarize", "translate")
summary_builder.add_edge("translate", END)
# 서브그래프를 독립적으로 컴파일
summary_subgraph = summary_builder.compile()

### 2.2 부모 그래프의 State와 노드

`summarize_document`는 부모 그래프의 노드입니다. 부모 State의 `document`를 Subgraph State의 `text`에 넣어 `summary_subgraph`를 호출하고, Subgraph가 돌려준 `summary`를 부모 State에 담아 돌려줘야 합니다. 이 셀은 함수만 만들고 아무것도 출력하지 않습니다.

In [ ]:
# ── 부모 그래프 ───────────────────────────────────────────────────────────────


class ParentState(TypedDict):
    messages: Annotated[list, add_messages]
    document: str
    summary: str


def fetch_document(state: ParentState) -> dict:
    # 실제로는 DB나 파일에서 문서를 가져온다.
    doc = "인공지능은 인간의 지능을 모방하는 컴퓨터 시스템으로, 최근 LLM의 등장으로 급격히 발전했습니다. 이제 AI는 텍스트 생성, 번역, 코딩 등 다양한 작업을 수행할 수 있습니다."
    return {"document": doc}


def summarize_document(state: ParentState) -> dict:
    # 서브그래프를 노드처럼 호출
    response = summary_subgraph.invoke({"text": state["document"], "summary": ""})
    return {"summary": response["summary"]}


def respond_node(state: ParentState) -> dict:
    response = llm.invoke(
        f"문서 요약(영문): {state['summary']}\n\n이를 바탕으로 사용자 질문에 답해줘: {state['messages'][-1].content}"
    )
    return {"messages": [response]}

### 2.3 부모 그래프 연결

부모 그래프의 노드 세 개는 등록되어 있습니다. 문서를 가져오고(`fetch_document`), Subgraph로 요약·번역하고(`summarize_document`), 요약을 바탕으로 답하는(`respond`) 순서로 잇습니다. 셀 마지막의 `app = parent_builder.compile()` 줄이 부모 그래프를 컴파일합니다.

In [ ]:
parent_builder = StateGraph(ParentState)
parent_builder.add_node("fetch_document", fetch_document)
parent_builder.add_node("summarize_document", summarize_document)  # 서브그래프 노드
parent_builder.add_node("respond", respond_node)

parent_builder.add_edge(START, "fetch_document")
parent_builder.add_edge("fetch_document", "summarize_document")
parent_builder.add_edge("summarize_document", "respond")
parent_builder.add_edge("respond", END)

app = parent_builder.compile()

### 2.4 실행하기

In [ ]:
parent_graph_result = app.invoke(
    {
        "messages": [HumanMessage(content="문서의 핵심 내용이 뭐야?")],
        "document": "",
        "summary": "",
    }
)

print(f"영문 요약: {parent_graph_result['summary']}")
results = parent_graph_result["messages"]
for m in results:
    m.pretty_print()

**볼 지점**
- `영문 요약:`은 Subgraph가 만든 값입니다. 원문은 한국어 두 문장이지만, Subgraph 안에서 요약 → 번역을 거쳐 영어 한 문장이 되었습니다.
- 부모 그래프는 Subgraph 안에 노드가 몇 개 있는지 모릅니다. `summarize_document` 노드 하나로 보일 뿐입니다. 같은 요약·번역 흐름이 다른 그래프에서도 필요하면 `summary_subgraph`를 그대로 가져다 씁니다.
- `Ai Message` 칸의 답은 영문 요약을 바탕으로 만든 답입니다. 원문 대신 요약을 넘겼으므로, 원문의 세부 내용은 답에서 빠질 수 있습니다.

## 3. MemorySaver로 대화 이어 가기

간단한 챗봇 그래프를 만들고 체크포인터 `MemorySaver`를 붙입니다. 같은 `thread_id`로 두 번 호출해, 두 번째 호출이 첫 호출에서 들은 이름을 기억하는지 봅니다.

결과는 맨 아래 "MemorySaver로 컴파일하고 실행하기" 셀에서 나옵니다.

준비 코드입니다. 도구를 불러오고 `llm = get_llm()`으로 모델을 준비합니다. 실행만 하면 됩니다.

In [ ]:
import os
import sqlite3
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

### 3.1 State와 chatbot 노드

`chatbot` 노드는 역할을 정한 시스템 메시지와 지금까지 쌓인 대화(`messages`)를 함께 모델에 보내고, 받은 답을 `messages`에 더해 돌려줘야 합니다. 체크포인터가 있으면 앞 호출의 메시지도 `messages`에 들어 있습니다.

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]


def chatbot(state: State) -> dict:
    system = SystemMessage(content="너는 친절한 AI 어시스턴트야.")
    response = llm.invoke([system] + state["messages"])
    return {"messages": [response]}

### 3.2 그래프 연결

노드가 하나뿐인 그래프입니다. `chatbot` 노드를 등록하고, 시작하면 chatbot으로 가고 chatbot이 끝나면 그래프도 끝나게 잇습니다.

In [ ]:
builder = StateGraph(State)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END);

### 3.3 MemorySaver로 컴파일하고 실행하기

**✍️ CHANGE_HERE: 두 번째 질문만 다른 thread_id로 보내면 AI가 이름을 기억할까요?**

**바꿔 볼 값** — 바로 아래 줄의 `"demo"` 자리에 복사해 붙여 넣기

```text
"demo_2"
```

이 셀만 다시 실행합니다. 확인했으면 `"demo"`로 되돌립니다.

그래프를 컴파일할 때 체크포인터를 넘기면, 호출이 끝날 때마다 State가 `thread_id`별로 저장됩니다. 아래 코드는 같은 `config`(thread_id `demo`)로 두 번 호출합니다. 첫 번째는 이름을 알려 주고, 두 번째는 이름을 묻습니다.

In [ ]:
# ── MemorySaver vs SqliteSaver ────────────────────────────────────────────────
#
# MemorySaver  : 프로세스 메모리에 저장 → 프로그램 재시작 시 기록 사라짐 (개발/데모용)
# SqliteSaver  : SQLite 파일에 저장   → 재시작해도 기록 유지 (로컬 프로토타입용)
# PostgresSaver: PostgreSQL에 저장    → 실제 서비스용 (langgraph-checkpoint-postgres 패키지)

print("=== 1. MemorySaver (휘발성) ===")
mem_app = builder.compile(checkpointer=MemorySaver())
config = {"configurable": {"thread_id": "demo"}}

mem_app.invoke(
    {"messages": [HumanMessage(content="내 이름은 홍길동이야.")]}, config=config
)
# 두 번째 질문을 보낼 대화(thread)
# ✍️ CHANGE_HERE: 두 번째 질문만 다른 thread_id로 보내면 AI가 이름을 기억할까요?
second_config = {"configurable": {"thread_id": "demo"}}
memory_graph_result = mem_app.invoke(
    {"messages": [HumanMessage(content="내 이름이 뭐야?")]}, config=second_config
)
results = memory_graph_result["messages"]
for m in results:
    m.pretty_print()

<details><summary>무슨 일이 생기나</summary>

두 번째 질문의 답이 "아직 이름을 알려주지 않았어"처럼 바뀌고, 출력에는 `Human Message`·`Ai Message`가 한 쌍만 남습니다(첫 질문이 보이지 않습니다).

체크포인터는 `thread_id`마다 대화를 따로 저장합니다. `demo_2`는 새 대화라서 `demo`에 저장된 첫 질문("내 이름은 홍길동이야.")을 모릅니다.

</details>

**볼 지점**
- 두 번째 호출에는 "내 이름이 뭐야?"만 보냈는데 "홍길동"이라고 답합니다. 체크포인터가 첫 호출의 메시지를 thread `demo`에 저장해 두었다가 이어 주었기 때문입니다.
- `MemorySaver`는 이 노트북의 메모리에 저장합니다. 커널을 다시 시작하면 기록이 사라집니다. 다음 장의 `SqliteSaver`는 파일에 저장합니다.

## 4. SqliteSaver로 저장하고 되돌아가기

같은 챗봇 그래프를 이번에는 `SqliteSaver`로 컴파일합니다. 대화 기록이 노트북과 같은 폴더의 `checkpoints.db` 파일에 저장되므로 커널을 다시 시작해도 남습니다.

그다음 저장된 체크포인트 목록을 보고, 그중 한 시점으로 되돌아가 다른 질문으로 다시 실행합니다(Time-Travel). 체크포인트는 그래프가 한 단계를 지날 때마다 하나씩 쌓이는 State 사진입니다.

코드는 완성되어 있습니다. 먼저 차례로 실행하고, 마지막 셀의 `# ✍️ CHANGE_HERE:` 줄을 바꿔 다시 실행합니다.

이 장의 준비 코드입니다. 도구를 불러오고, 앞 장에서 만든 것과 같은 State·chatbot 노드·그래프 연결을 완성된 코드로 다시 준비합니다. 펼치지 않고 실행만 합니다.

In [ ]:
import os
import sqlite3
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()


class State(TypedDict):
    messages: Annotated[list, add_messages]


def chatbot(state: State) -> dict:
    system = SystemMessage(content="너는 친절한 AI 어시스턴트야.")
    return {"messages": [llm.invoke([system] + state["messages"])]}


builder = StateGraph(State)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END);

### 4.1 SqliteSaver로 저장하기

In [ ]:
print("=== 2. SqliteSaver (영속성) ===")
DB_PATH = "checkpoints.db"  # 노트북과 같은 폴더에 만든다

conn = sqlite3.connect(DB_PATH, check_same_thread=False)
# SqliteSaver 체크포인터로 컴파일한다
sqlite_app = builder.compile(checkpointer=SqliteSaver(conn))
config_sql = {"configurable": {"thread_id": "persistent_demo"}}

sqlite_app.invoke(
    {"messages": [HumanMessage(content="내 취미는 독서야.")]}, config=config_sql
)
sqlite_graph_result = sqlite_app.invoke(
    {"messages": [HumanMessage(content="내 취미가 뭐야?")]}, config=config_sql
)
results = sqlite_graph_result["messages"]
for m in results:
    m.pretty_print()

### 4.2 체크포인트 목록 보기

In [ ]:
# 저장된 체크포인트 목록 확인 (0번이 가장 최근)
checkpoints = list(sqlite_app.get_state_history(config_sql))
print(f"저장된 체크포인트 수: {len(checkpoints)}")
for i, cp in enumerate(checkpoints):
    print(
        f"  [{i}] {cp.config['configurable']['checkpoint_id'][:8]}... | 메시지 수: {len(cp.values.get('messages', []))}"
    )

### 4.3 과거 체크포인트로 되돌아가기

**✍️ CHANGE_HERE: 되돌아갈 시점을 목록의 맨 끝(가장 처음 체크포인트)으로 바꾸면 AI는 취미를 알까요?**

**바꿔 볼 값** — 바로 아래 줄의 `[0]` 자리에 복사해 붙여 넣기

```text
[-1]
```

목록의 0번은 가장 최근, 맨 끝(`-1`)은 가장 처음 시점입니다. 이 셀만 다시 실행합니다(목록 셀은 다시 실행하지 않아도 됩니다). 확인했으면 `[0]`으로 되돌립니다.

In [ ]:
# Time-Travel: 특정 체크포인트로 되돌아가 다른 흐름으로 재실행
# ✍️ CHANGE_HERE: 되돌아갈 시점을 목록의 맨 끝(가장 처음 체크포인트)으로 바꾸면 AI는 취미를 알까요?
past_config = checkpoints[0].config
print("=== Time-Travel: 과거 시점에서 다른 질문으로 재실행 ===")
time_travel_result = sqlite_app.invoke(
    {"messages": [HumanMessage(content="내 취미가 뭐라고 했더라?")]},
    config=past_config,
)
results = time_travel_result["messages"]
for m in results:
    m.pretty_print()

<details><summary>무슨 일이 생기나</summary>

출력에 방금 보낸 질문과 답 한 쌍만 나오고, AI가 "아직 취미를 말씀해 주신 적이 없다"고 답합니다. `[0]`일 때는 앞의 대화가 모두 이어져 "독서"라고 답했습니다.

목록의 맨 끝은 대화를 시작하기 전(메시지 수 0)의 State입니다. 그 시점으로 돌아가 다시 실행하면 그 뒤에 쌓인 대화는 보이지 않습니다. 원래 기록은 지워지지 않고, 그 시점에서 새 갈래가 생깁니다.

</details>

**볼 지점**
- 첫 셀을 다시 실행하면 같은 thread(`persistent_demo`)에 기록이 계속 쌓여 체크포인트 수가 늘어납니다. 기록이 파일에 남는다는 증거입니다. 커널을 다시 시작해도 마찬가지입니다.
- 되돌아간 시점에서 다시 실행하면 원래 기록은 그대로 두고 그 시점에서 새 갈래가 생깁니다. 되돌아가기 셀을 실행한 뒤 목록 셀을 다시 실행하면 체크포인트가 늘어난 것을 볼 수 있습니다.
- 처음 상태로 다시 해 보고 싶으면 먼저 커널을 다시 시작합니다(파일이 열려 있으면 Windows에서 지워지지 않습니다). 그다음 노트북과 같은 폴더의 `checkpoints.db`와, 있으면 `checkpoints.db-wal`·`checkpoints.db-shm`까지 세 파일을 지우고 이 장의 셀을 처음부터 실행합니다.

## 5. Memory Store로 대화를 넘어 기억하기

체크포인터는 thread 하나 안에서만 대화를 이어 줍니다. 이번에는 thread를 넘어 남는 장기 기억 저장소(`InMemoryStore`)를 씁니다. 사용자가 이름이나 취미를 말하면 모델이 답 앞에 `[기억 저장: …]`을 붙이고, 코드는 그 내용을 사용자별 namespace(`("memories", user_id)`)에 저장합니다. 다음 대화에서는 저장된 기억을 시스템 메시지에 넣어 줍니다.

코드는 완성되어 있습니다. 먼저 차례로 끝까지 실행하고, `# ✍️ CHANGE_HERE:` 줄 두 곳을 바꿔 다시 실행합니다.

도구와 모델을 준비하는 코드입니다. 실행만 하면 됩니다.

In [ ]:
import re
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.store.memory import InMemoryStore
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

### 5.1 장기 기억 Store 만들기

In [ ]:
# ── Checkpointer vs Store ─────────────────────────────────────────────────────
#
# Checkpointer (MemorySaver 등)
#   - thread(세션) 단위로 State를 저장
#   - 같은 thread 안에서 대화 흐름을 이어감 (단기 기억)
#
# Store (InMemoryStore 등)
#   - thread를 넘어 여러 세션에서 공유되는 장기 기억
#   - 사용자 프로필, 선호도, 중요 사실 등을 저장
#   - namespace (tuple)로 데이터를 논리적으로 분리한다

# 장기 기억을 저장할 Store를 만든다
store = InMemoryStore()

### 5.2 기억을 읽고 쓰는 chatbot 노드

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    user_id: str


def chatbot(state: State) -> dict:
    user_id = state.get("user_id", "anonymous")

    # Store에서 이 사용자의 장기 기억 조회
    namespace = ("memories", user_id)
    memories = store.search(namespace)
    memory_text = (
        "\n".join(f"- {m.value['fact']}" for m in memories) if memories else "없음"
    )

    system_prompt = f"""너는 친절한 AI 어시스턴트야.

사용자에 대해 알고 있는 정보:
{memory_text}

대화 중 사용자의 중요한 정보(이름, 취미, 직업 등)가 나오면,
"[기억 저장: <내용>]" 형식으로 메시지 앞에 표시해줘."""

    response = llm.invoke([SystemMessage(content=system_prompt)] + state["messages"])

    # LLM이 기억할 내용을 표시했으면 Store에 저장
    content = response.content
    if "[기억 저장:" in content:
        facts = re.findall(r"\[기억 저장: (.+?)\]", content)
        for i, fact in enumerate(facts):
            key = f"fact_{len(memories) + i}"
            store.put(namespace, key, {"fact": fact})

    return {"messages": [response]}

### 5.3 그래프 연결과 컴파일

In [ ]:
builder = StateGraph(State)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)
# MemorySaver 체크포인터와 Store를 함께 넘겨 컴파일한다
app = builder.compile(checkpointer=MemorySaver(), store=store)

### 5.4 대화 함수

In [ ]:
def chat(user_id: str, thread_id: str, message: str) -> str:
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke(
        {"messages": [HumanMessage(content=message)], "user_id": user_id},
        config=config,
    )
    return result["messages"][-1].content

### 5.5 두 세션으로 대화하기

**✍️ CHANGE_HERE: 세션 2를 다른 사용자(`user_002`)로 바꾸면 이름과 취미를 알까요?**

**바꿔 볼 값** — 바로 아래 줄의 `"user_001"` 자리에 복사해 붙여 넣기

```text
"user_002"
```

바꾼 뒤 5.1 셀부터 차례로 다시 실행합니다(대화 기록과 기억을 처음부터 다시 쌓기 위해서입니다). 확인했으면 `"user_001"`로 되돌립니다.

In [ ]:
print("=== 세션 1: 정보 입력 ===")
print(chat("user_001", "session_1", "안녕! 나는 이지수야. 취미는 피아노 연주야."))

print("\n=== 세션 2: 새 세션에서도 기억 유지 ===")
# ✍️ CHANGE_HERE: 세션 2를 다른 사용자(`user_002`)로 바꾸면 이름과 취미를 알까요?
print(chat("user_001", "session_2", "내 이름이랑 취미 알아?"))

print("\n=== Store에 저장된 장기 기억 ===")
memories = store.search(("memories", "user_001"))
for m in memories:
    print(f"  [{m.key}] {m.value['fact']}")

<details><summary>무슨 일이 생기나</summary>

세션 2가 "아직 알려주지 않았어! 이름이 뭐야?"처럼 답해 이름과 취미를 모릅니다. 맨 아래 `Store에 저장된 장기 기억`에는 `user_001`의 기억(이지수, 피아노 연주)이 그대로 있습니다.

기억은 `("memories", user_id)` 이름표(namespace)로 사용자마다 따로 저장됩니다. `user_002`의 이름표에는 아무것도 없어서, 시스템 메시지에 넣을 기억이 없습니다.

</details>

**볼 지점**
- 세션 2는 새 thread(`session_2`)라 체크포인터에는 세션 1의 대화가 없습니다. 그런데도 이름과 취미를 압니다. 시스템 메시지에 Store의 기억이 들어갔기 때문입니다.
- 기억은 `("memories", user_id)` namespace에 사용자별로 저장됩니다. 마지막 줄들이 `user_001`의 기억을 꺼내 보여 줍니다.
- `[기억 저장: …]` 표시는 사용자에게 보이는 답에도 그대로 남습니다. 저장이 모델의 형식 준수에 달려 있다는 점도 이 코드의 약점입니다.

## 6. 장점과 단점을 동시에 정리하기

한 주제(Remote Work)의 장점과 단점을 두 노드가 동시에 정리하고, 세 번째 노드가 둘을 모아 한 문장 결론을 냅니다. START에서 두 노드로 갈라지는 것이 fan-out, 두 노드가 한 노드로 모이는 것이 fan-in입니다.

결과는 맨 아래 "실행하기" 셀에서 나옵니다.

이 장의 준비 코드입니다. 앞 장들과 같이 도구와 모델을 준비합니다. 펼치지 않고 실행만 합니다.

In [ ]:
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.types import Send
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

### 6.1 장점·단점·결론 노드

In [ ]:
# ── 패턴 1: 정적 병렬 (Fan-out → Fan-in) ────────────────────────────────────
# 고정된 수의 노드를 병렬로 실행할 때 사용한다.
# 각 분기가 자기 필드(pros, cons)에 결과를 쓰고, 이후 하나의 노드가 합쳐서 처리한다.


class StaticState(TypedDict):
    topic: str
    pros: str
    cons: str
    final: str


def pros_node(state: StaticState) -> dict:
    result = llm.invoke(f"{state['topic']}의 장점 3가지를 간결하게 나열해줘.")
    return {"pros": result.content}


def cons_node(state: StaticState) -> dict:
    result = llm.invoke(f"{state['topic']}의 단점 3가지를 간결하게 나열해줘.")
    return {"cons": result.content}


def summary_node(state: StaticState) -> dict:
    result = llm.invoke(
        f"장점:\n{state['pros']}\n\n단점:\n{state['cons']}\n\n위 내용을 바탕으로 한 문장 결론을 내려줘."
    )
    return {"final": result.content}

### 6.2 병렬로 연결하기

노드 세 개(`pros`, `cons`, `summary`)는 등록되어 있습니다. START에서 `pros`와 `cons`가 함께 시작하고, 두 노드가 모두 끝나면 `summary`가 결론을 내고, 그다음 그래프가 끝나게 잇습니다. 셀 마지막의 `static_app = static_builder.compile()` 줄이 그래프를 컴파일합니다.

In [ ]:
static_builder = StateGraph(StaticState)
static_builder.add_node("pros", pros_node)
static_builder.add_node("cons", cons_node)
static_builder.add_node("summary", summary_node)

static_builder.add_edge(START, "pros")
static_builder.add_edge(START, "cons")
static_builder.add_edge("pros", "summary")
static_builder.add_edge("cons", "summary")
static_builder.add_edge("summary", END)
static_app = static_builder.compile()

### 6.3 실행하기

In [ ]:
print("=== 정적 병렬 (Fan-out → Fan-in) ===")
result = static_app.invoke(
    {"topic": "Remote Work", "pros": "", "cons": "", "final": ""}
)
print(f"장점: {result['pros'][:80]}...")
print(f"단점: {result['cons'][:80]}...")
print(f"결론: {result['final']}")
print()

**볼 지점**
- `장점:`과 `단점:`은 앞 80자만 출력합니다. 두 노드는 서로의 결과를 보지 않고 각자 주제만 받아 정리합니다. 그래서 동시에 실행할 수 있습니다.
- `결론:`은 두 결과를 모두 받은 `summary` 노드가 냅니다. `summary`로 들어오는 엣지가 두 개이므로 두 노드가 모두 끝난 뒤에 실행됩니다.
- 두 노드는 각자 `pros`, `cons`라는 다른 필드에 씁니다. 같은 필드에 쓰지 않으므로 합치는 규칙(reducer)이 필요 없습니다.

## 7. 질문 수만큼 동시에 답하기

이번에는 질문 목록을 받아, 질문 수만큼 worker를 만들어 동시에 답하게 합니다. 질문이 몇 개일지 그래프를 만들 때는 모르므로, 엣지 대신 `Send`로 실행할 때 작업을 만듭니다. `fan_out` 함수가 질문마다 `Send("worker", {...})`를 하나씩 만들어 목록으로 돌려주면, LangGraph가 그 수만큼 `worker` 노드를 병렬로 실행합니다.

worker마다 결과를 `results` 목록에 더합니다. 여러 worker가 같은 필드에 쓰므로, State의 `results`에는 두 목록을 이어 붙이는 규칙(`Annotated[..., lambda a, b: a + b]`)이 붙어 있습니다.

결과는 맨 아래 "실행하기" 셀에서 나옵니다.

준비 코드는 앞 장과 같습니다. 실행만 하면 됩니다.

In [ ]:
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.types import Send
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

### 7.1 State와 worker 노드

In [ ]:
# ── 패턴 2: 동적 병렬 (Send API) ────────────────────────────────────────────
# 실행 시점에 작업 수가 결정될 때 사용한다.
# Send(노드명, state)를 리스트로 반환하면 LangGraph가 해당 노드를 병렬로 실행한다.


class DynamicState(TypedDict):
    queries: list[str]
    results: Annotated[list[str], lambda a, b: a + b]  # 결과를 누적


class WorkerState(TypedDict):
    query: str
    results: Annotated[list[str], lambda a, b: a + b]


def worker_node(state: WorkerState) -> dict:
    """각 쿼리를 독립적으로 처리하는 워커 노드."""
    result = llm.invoke(f'"{state["query"]}"에 대해 한 문장으로 답해줘.')
    return {"results": [f"[{state['query']}] {result.content}"]}

### 7.2 fan_out 함수

`fan_out`은 State의 `queries`(질문 목록)를 보고, 질문 하나마다 `worker` 노드로 보낼 `Send`를 하나씩 만들어 목록으로 돌려줘야 합니다. `Send`에 담는 딕셔너리는 worker가 받을 State(`WorkerState`)의 모양을 따릅니다. 이 셀은 함수만 만들고 아무것도 출력하지 않습니다.

In [ ]:
def fan_out(state: DynamicState) -> list[Send]:
    """쿼리 수만큼 worker_node를 병렬로 생성해 보낸다."""
    return [Send("worker", {"query": q, "results": []}) for q in state["queries"]]

### 7.3 동적으로 연결하기

`worker` 노드는 등록되어 있습니다. 시작하면 `fan_out`이 돌려준 `Send` 목록대로 worker를 만들고, worker가 끝나면 그래프가 끝나게 잇습니다. `Send` 목록을 돌려주는 함수는 조건부 엣지에 넣습니다. 셀 마지막의 `dynamic_app = dynamic_builder.compile()` 줄이 그래프를 컴파일합니다.

In [ ]:
dynamic_builder = StateGraph(DynamicState)
dynamic_builder.add_node("worker", worker_node)
dynamic_builder.add_conditional_edges(START, fan_out, ["worker"])
dynamic_builder.add_edge("worker", END)
dynamic_app = dynamic_builder.compile()

### 7.4 실행하기

**✍️ CHANGE_HERE: 질문을 하나 더 넣으면 worker는 몇 번 실행될까요?**

**바꿔 볼 값** — 바로 아래 줄의 `"MCP란?"]` 자리에 복사해 붙여 넣기

```text
"MCP란?", "A2A란?"]
```

이 셀만 다시 실행합니다. 확인했으면 질문 목록을 되돌립니다.

In [ ]:
print("=== 동적 병렬 (Send API) ===")
# ✍️ CHANGE_HERE: 질문을 하나 더 넣으면 worker는 몇 번 실행될까요?
queries = ["LangGraph란?", "RAG란?", "MCP란?"]
result2 = dynamic_app.invoke({"queries": queries, "results": []})
for r in result2["results"]:
    print(f"  {r}")

<details><summary>무슨 일이 생기나</summary>

결과 줄이 4개가 되고 `[A2A란?]` 줄이 생깁니다. 그래프 코드는 그대로인데 `worker`가 질문 수만큼 4번 실행됐습니다.

몇 번 실행할지는 실행할 때 `fan_out`이 돌려준 `Send` 목록의 길이로 정해집니다. 답의 내용은 검색 없이 모델이 아는 것으로 쓰므로 `A2A`처럼 새 용어는 엉뚱하게 풀기도 합니다(예: "Ask to Answer").

</details>

**볼 지점**
- 그래프에는 `worker` 노드가 하나뿐인데, 질문 수만큼 실행되었습니다. 몇 번 실행할지는 그래프를 만들 때가 아니라 실행할 때 `fan_out`이 정합니다.
- worker들은 동시에 실행되므로, 질문을 늘려도 걸리는 시간은 질문 수만큼 늘지 않습니다.
- 모든 worker가 같은 `results`에 결과를 더했는데도 줄이 모두 남았습니다. 왜 그런지는 "State와 worker 노드" 셀에서 `results`에 붙은 `Annotated[..., lambda a, b: a + b]`를 보면 알 수 있습니다. 모르겠으면 그 셀을 AI에게 붙여 넣고 물어봅니다.
- 답의 내용은 검색 없이 모델이 아는 것으로 씁니다. `MCP`처럼 새 용어는 엉뚱하게 풀기도 합니다(예: "Microsoft Certified Professional"). 답을 그대로 믿지 말고 확인하는 것도 실습입니다.

## 8. 정리

LangGraph로 조금 더 큰 그래프를 만드는 방법을 여섯 장에 걸쳐 실행해 보았습니다.

| 장 | 한 일 | 확인한 것 |
|---|---|---|
| 2. Subgraph | 요약·번역 그래프를 부모 노드 안에서 호출했습니다 | State가 달라도 노드 함수가 값을 옮겨 주면 됩니다 |
| 3. MemorySaver | 같은 thread_id로 두 번 호출했습니다 | 체크포인터가 앞 대화를 이어 줍니다 |
| 4. SqliteSaver | 파일에 저장하고 지난 시점으로 되돌아갔습니다 | 기록이 파일에 남고, 과거 체크포인트에서 새 갈래로 다시 실행할 수 있습니다 |
| 5. Memory Store | 새 thread에서도 사용자 정보를 기억하게 했습니다 | 장기 기억은 namespace로 사용자별로 나뉩니다. 노드가 어떤 store를 쓰는지는 코드를 봐야 압니다 |
| 6. Fan-out/Fan-in | 장점·단점을 동시에 정리하고 합쳤습니다 | 병렬인지는 출력이 아니라 엣지를 봐야 알 수 있습니다 |
| 7. Send | 질문 수만큼 worker를 만들어 동시에 답하게 했습니다 | 작업 수를 실행할 때 정하고, 결과는 reducer로 모읍니다 |

값을 바꿔 다시 실행하며 Subgraph에 넘기는 문서, `thread_id`, 되돌아갈 체크포인트, 사용자(namespace)와 compile의 `store`, 병렬 작업 수가 그래프의 동작을 어떻게 바꾸는지 확인했습니다.